# ClearSpace Image Generation Service

This notebook launches the optional **AI Visual Preview** service used by
ClearSpace. It loads Stable Diffusion 1.5, depth ControlNet, and MiDaS on
a Colab GPU, starts the FastAPI service, and exposes it through ngrok.

For architecture, configuration, evaluation findings, and troubleshooting,
see `colab_service/README.md`. The AI Visual Preview is illustrative and
should not be treated as a precise or item-preserving reorganisation.

## Before you start

Attach a Colab GPU and configure these secrets before running the notebook:

| Secret | Purpose |
|---|---|
| `GITHUB_TOKEN` | Clone the repository |
| `NGROK_AUTHTOKEN` | Authenticate ngrok |
| `NGROK_DOMAIN` | Bind the configured ngrok domain |

**Security:** Keep credentials and the service URL private. The ngrok endpoint
has no application-level authentication, so run it only while actively testing
and terminate the Colab runtime afterwards.


## Step 1 — Check GPU

Verify that the Colab runtime has CUDA available before installing dependencies or loading models.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. In Colab: Runtime > Change runtime type > "
        "Hardware accelerator > GPU, then Runtime > Restart runtime and "
        "re-run this notebook from the top. (VS Code Colab extension: "
        "make sure you're connected to a GPU-backed runtime.)"
    )

print(f"CUDA GPU available: {torch.cuda.get_device_name(0)}")


## Step 2 — Load secrets

Load the required credentials from Colab Secrets. If a value is unavailable,
the notebook prompts for it securely.

In [ ]:
import getpass
import os


def _get_secret(name: str) -> str:
    """Reads a secret by name. Tries Colab Web's Secrets panel first
    (google.colab.userdata) -- the normal path. Falls back to a masked,
    interactive prompt if userdata is unavailable (e.g. the VS Code Colab
    extension, which does not always expose userdata.get()) or the named
    secret isn't set there. The typed value is never echoed, never
    logged, and never written to a file. A blank (empty or
    whitespace-only) value from EITHER path is rejected with a clear
    error rather than silently proceeding -- a blank secret is never a
    valid one."""
    try:
        from google.colab import userdata

        value = userdata.get(name)
        if value and value.strip():
            return value
    except Exception:
        pass  # userdata unavailable (e.g. VS Code Colab extension) or secret not set there
    value = getpass.getpass(f"Enter {name} (input hidden, not stored): ")
    if not value or not value.strip():
        raise RuntimeError(f"{name} was blank -- a real, non-blank secret value is required")
    return value


GITHUB_TOKEN = _get_secret("GITHUB_TOKEN")
NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
NGROK_DOMAIN = _get_secret("NGROK_DOMAIN")

# Only the two ngrok values are needed as plain environment variables by
# the rest of this notebook / colab_service.config.Settings -- GITHUB_TOKEN
# is used once, transiently, by the clone cell below, and is never stored
# in os.environ at all.
os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTHTOKEN
os.environ["NGROK_DOMAIN"] = NGROK_DOMAIN

print("Secrets loaded (values not shown).")


## Step 3 — Clone repository

Clone the ClearSpace repository into the Colab runtime using the configured
GitHub credential. A fresh runtime is expected; if the repository directory
already exists, the cell stops instead of replacing or reusing it.


In [ ]:
import base64
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/yxnmei/clearspace-fyp.git"
REPO_DIR = Path("/content/clearspace-fyp")
GITHUB_USERNAME = "yxnmei"

try:
    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} already exists. This notebook never reuses a "
            "possibly-stale checkout and never deletes one automatically -- "
            "either restart the Colab runtime (Runtime > Restart runtime) "
            "for a guaranteed-clean session, or delete that directory "
            "yourself first if you are sure it is safe to discard."
        )

    # GitHub HTTPS git uses the personal access token as the HTTP Basic
    # password. Neither GITHUB_TOKEN nor the encoded credential is ever
    # printed, placed in the clone URL, or passed as a command argument
    # -- both travel only inside this one subprocess's own environment,
    # via GIT_CONFIG_VALUE_0 below.
    credential = base64.b64encode(f"{GITHUB_USERNAME}:{GITHUB_TOKEN}".encode("utf-8")).decode("ascii")

    # Built from a FILTERED copy of os.environ, not the raw dict:
    # NGROK_AUTHTOKEN/NGROK_DOMAIN are already plain env vars by this
    # point (set in the secrets cell above) and have no reason to be
    # visible to the clone subprocess at all.
    clone_env = {k: v for k, v in os.environ.items() if k not in ("NGROK_AUTHTOKEN", "NGROK_DOMAIN")}
    clone_env["GIT_CONFIG_COUNT"] = "1"
    clone_env["GIT_CONFIG_KEY_0"] = "http.extraHeader"
    clone_env["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {credential}"

    try:
        subprocess.run(
            ["git", "clone", REPO_URL, str(REPO_DIR)],
            check=True, capture_output=True, text=True, env=clone_env,
        )
    except subprocess.CalledProcessError:
        # Deliberately never prints exc.stderr/exc.stdout, and raises with
        # `from None` -- even though the token is not in argv/URL here,
        # nothing about a git error's own text is trusted not to echo the
        # Authorization header value back in some edge case (e.g. a
        # misconfigured proxy). One fixed, sanitized message only.
        raise RuntimeError(
            "git clone failed -- check repository access and your "
            "GITHUB_TOKEN secret, then try again"
        ) from None
    finally:
        # Discard the temporary token-bearing environment and the
        # encoded credential immediately -- neither is needed again
        # this session.
        del clone_env
        del credential
finally:
    # Cleared on EVERY exit path from this cell, success or failure --
    # including the already-exists check above raising before a clone
    # was ever attempted -- so no failure path leaves the token live in
    # notebook memory.
    GITHUB_TOKEN = None

print(f"Cloned to {REPO_DIR} (origin remote was never given the token -- clean from the start).")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"{REPO_DIR} added to sys.path.")


## Step 4 — Install dependencies

Install the image-generation service dependencies from `colab_service/requirements.txt`.

In [ ]:
get_ipython().system('pip install -q -r {REPO_DIR}/colab_service/requirements.txt')

from importlib.metadata import version as _pkg_version

print("colab_service dependencies installed.")
# Print the installed diffusers version (no secrets) right after the
# install -- e.g. 0.40.0, whose StableDiffusionControlNetImg2ImgPipeline
# did not expose the pipeline-level enable_vae_slicing() method in the
# verified Colab runtime; load_pipeline() prefers the current
# pipe.vae.enable_slicing() API and falls back to that legacy method
# (see colab_service/pipeline.py).
print("diffusers version:", _pkg_version("diffusers"))


## Step 5 — Load models

Load Stable Diffusion 1.5, depth ControlNet, and MiDaS onto the GPU. The
first run may take several minutes while model weights are downloaded.

In [ ]:
import colab_service.app as app_module

print("Loading models -- this can take several minutes on first run (fresh weights download).")
app_module.load_models()
print(
    "Models loaded. base_model:", app_module._state.base_model_id,
    "| controlnet_model:", app_module._state.controlnet_model_id,
)


## Step 6 — Start service

Start the FastAPI service, verify its health endpoint, and expose it through
the configured ngrok tunnel. When ready, the cell prints the public service
URL; use it as `IMAGE_GEN_BASE_URL` in the ClearSpace backend.

If either the local or public health check fails, the cell stops with an error.


In [ ]:
import os
import threading
import time

import requests
from pyngrok import ngrok

_settings = app_module.config.get_settings()


def _run_server():
    import uvicorn

    uvicorn.run(app_module.app, host=_settings.host, port=_settings.port, log_level="warning")


def _is_compatible_health_body(data: object) -> bool:
    """Mirrors app.models.image_gen_client.check_health()'s exact
    compatibility criteria on the JSON body (status/api_version/
    service_version/capabilities.depth_controlnet) -- see that
    function's own docstring in backend/. The HTTP status-code gate
    (2xx) is checked separately by the caller, _poll_health, before
    this is ever called. A {"status": "loading"} body (returned by
    colab_service/app.py's own /health handler before load_models()
    has finished) deliberately does NOT satisfy this, so the poll
    keeps retrying through it instead of treating a not-yet-ready
    service as ready."""
    if not isinstance(data, dict):
        return False
    if data.get("status") != "ok":
        return False
    if data.get("api_version") != app_module.schemas.API_VERSION:
        return False
    service_version = data.get("service_version")
    if not isinstance(service_version, str) or not service_version.strip():
        return False
    capabilities = data.get("capabilities")
    if not isinstance(capabilities, dict):
        return False
    if capabilities.get("depth_controlnet") is not True:
        return False
    return True


def _poll_health(base_url, *, description, attempts=20, delay_s=1.5, hint=""):
    """Bounded polling of ``{base_url}/health`` -- never a fixed
    sleep-and-hope wait. Retries a short, fixed number of times with a
    short delay, tolerating the brief window before the endpoint is
    reachable AND the longer, expected window where /health responds
    with {"status": "loading"} because load_models() has not finished
    yet. Returns only when ALL of: the HTTP status is 2xx, the body
    parses as JSON, the parsed JSON is an object, and that object is
    the full compatible "ok" contract shape (see
    _is_compatible_health_body). A non-2xx status (e.g. an offline
    ngrok agent's ERR_NGROK_3200 404), invalid JSON, a loading body,
    any other incompatible body, or a connection failure are all
    retryable and never printed raw. Raises one clear, sanitized
    error (no raw response body, URL, or exception text) if the poll
    budget is exhausted."""
    last_state = None
    for _ in range(attempts):
        try:
            response = requests.get(f"{base_url}/health", timeout=5)
        except requests.RequestException as exc:
            last_state = type(exc).__name__
            time.sleep(delay_s)
            continue
        if not (200 <= response.status_code < 300):
            last_state = f"HTTP {response.status_code}"
            time.sleep(delay_s)
            continue
        try:
            data = response.json()
        except ValueError as exc:
            last_state = type(exc).__name__
            time.sleep(delay_s)
            continue
        if _is_compatible_health_body(data):
            return data
        last_state = "incompatible health response"
        time.sleep(delay_s)
    message = (
        f"{description} /health did not report a compatible ready body after {attempts} "
        f"attempts ({attempts * delay_s:.0f}s) -- last observed state: {last_state}. The "
        "server thread or model loading may have failed; check for errors printed above this cell."
    )
    if hint:
        message = f"{message} {hint}"
    raise RuntimeError(message)


def _poll_local_health():
    """Poll the LOCAL service until it reports the full compatible
    ready body -- a thin wrapper over _poll_health kept as a stable
    name."""
    return _poll_health(f"http://127.0.0.1:{_settings.port}", description="Local")


server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

# Poll the LOCAL health endpoint until the service is genuinely, fully
# ready BEFORE opening the ngrok tunnel -- the public URL must never be
# exposed for a service that isn't ready yet.
local_health = _poll_local_health()

# Rebind the tunnel cleanly. ngrok.kill() drops any ngrok agent this
# pyngrok process is still managing (a harmless no-op on a fresh
# runtime), so a stale/offline agent cannot leave the reserved
# NGROK_DOMAIN answering with ERR_NGROK_3200; the fresh connect() below
# then rebinds it.
ngrok.kill()
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
tunnel = ngrok.connect(_settings.port, domain=os.environ["NGROK_DOMAIN"])

# Verify the PUBLIC tunnel against the SAME health contract before
# reporting the service ready -- a rebind that has not propagated yet,
# or a domain still held by another session, is caught here rather
# than handed to the backend as a broken URL.
public_health = _poll_health(
    tunnel.public_url,
    description="Public tunnel",
    hint=(
        "If this persists, the reserved domain is still bound to another, offline ngrok "
        "agent -- terminate every other Colab session using it, then restart this runtime "
        "and re-run from the top."
    ),
)

print("Service URL:", tunnel.public_url)
print("Local health check:", local_health)
print("Public health check:", public_health)


## Cleanup — Stop the service

When testing is complete:

1. **Stop the tunnel** by uncommenting and running the cleanup cell below.
2. **Terminate the Colab runtime**:
   - **Colab Web**: `Runtime > Manage sessions > Terminate`.
   - **VS Code Colab extension**: run **Colab: Remove Server** from the
     Command Palette.

Closing the browser or editor tab may leave the runtime and tunnel active.

In [ ]:
# Uncomment and run when you want to stop the tunnel (this alone does
# NOT terminate the Colab runtime -- see step 2 above, which you must
# also do, via the Colab Web or VS Code Colab extension UI):
#
# from pyngrok import ngrok
# ngrok.disconnect(tunnel.public_url)
# ngrok.kill()
